# Inverse Quadratic Interpolation (IQI)
Inverse Quadratic Interpolation is a root-finding method.
Instead of drawing a straight line through 2 points (like the Secant method), it draws a sideways parabola through the last 3 points we guessed. 
Because it's a sideways parabola (x as a function of y), we can just ask the parabola: "What is x when y=0?" to get our next guess.
It converges very fast, but requires 3 starting guesses.

### The Math Behind That Ugly Formula
If you do standard Lagrange interpolation to fit a sideways parabola (where $y$ is the input and $x$ is the output) through points $(x_0, y_0)$, $(x_1, y_1)$, and $(x_2, y_2)$, the math looks like this:

$$ P(y) = x_0 \frac{(y - y_1)(y - y_2)}{(y_0 - y_1)(y_0 - y_2)} + x_1 \frac{(y - y_0)(y - y_2)}{(y_1 - y_0)(y_1 - y_2)} + x_2 \frac{(y - y_0)(y - y_1)}{(y_2 - y_0)(y_2 - y_1)} $$

To find the next root, we plug in $y = 0$. 
However, if you just plug 0 into that raw formula, your computer might crash or lose precision because you will be dividing by very tiny differences (like $y_0 - y_1$ when the points are close together). 

To make the code safe and fast, Numerical Methods textbooks use an algebraic trick. We define ratios of the y-values:
* $q = \frac{y_0}{y_1}$
* $r = \frac{y_2}{y_1}$
* $s = \frac{y_2}{y_0}$

If you substitute $q$, $r$, and $s$ into the $P(0)$ equation and do the algebra to factor it, it perfectly simplifies into this beautiful, optimized formula:

$$ x_3 = x_2 - \frac{r(r - q)(x_2 - x_1) + s(1 - r)(x_2 - x_0)}{(q - 1)(r - 1)(s - 1)} $$

This is literally just the Lagrange formula evaluated at $y=0$, mathematically reorganized to prevent floating-point errors. This is exactly what the Python code executes.

In [3]:
import numpy as np

def f(x):
    return x**3 + x - 1

def iqi(f, x0, x1, x2, tol=1e-8, max_iter=100):
    for i in range(max_iter):
        y0, y1, y2 = f(x0), f(x1), f(x2)
        
        # Check if we are close enough to zero
        if abs(y2) < tol:
            return x2
            
        # These are the terms for the Lagrange interpolation formula (but sideways!)
        # We calculate the next x guess by plugging y=0 into the sideways parabola
        q = f(x0) / f(x1)
        r = f(x2) / f(x1)
        s = f(x2) / f(x0)
        
        # Formula for the next guess
        x3 = x2 - (r * (r - q) * (x2 - x1) + s * (1 - r) * (x2 - x0)) / ((q - 1) * (r - 1) * (s - 1))
        
        # Shift everything over for the next loop
        x0, x1, x2 = x1, x2, x3
        
    return x2

# Example usage
root = iqi(f, 0.1, 0.15, 0.25)
print("Root is:", root)

Root is: 0.682327800696779
